# 05 - Final Load Preparation

**Goal:** Prepare one clean Tableau-ready fact table and supporting KPI CSVs after EDA/statistical analysis.

This notebook is a handoff artifact for the Visualization Lead. It does not replace Tableau; it creates reliable inputs for Tableau.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ANALYSIS_DIR = PROCESSED_DIR / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

cleaned_paths = sorted(PROCESSED_DIR.glob('cleaned_hospital_*.csv'))
frames = [pd.read_csv(path) for path in cleaned_paths]
df = pd.concat(frames, ignore_index=True)
print(f'Loaded shape: {df.shape}')

In [ ]:
tableau_df = df.copy()

tableau_df['Recovered_Flag'] = (tableau_df['Outcome'] == 'Recovered').astype(int)
tableau_df['Deceased_Flag'] = (tableau_df['Outcome'] == 'Deceased').astype(int)
tableau_df['Under_Treatment_Flag'] = (tableau_df['Outcome'] == 'Under Treatment').astype(int)
tableau_df['Insured_Flag'] = (tableau_df['Insurance'] == 'Yes').astype(int)
tableau_df['Private_Hospital_Flag'] = (tableau_df['Hospital_Type'] == 'Private').astype(int)

overall_avg_cost = tableau_df['Treatment_Cost'].mean()
tableau_df['Cost_vs_Overall_Avg'] = tableau_df['Treatment_Cost'] - overall_avg_cost

diagnosis_avg = tableau_df.groupby('Diagnosis')['Treatment_Cost'].transform('mean')
tableau_df['Cost_vs_Diagnosis_Avg'] = tableau_df['Treatment_Cost'] - diagnosis_avg

tableau_df['Operational_Access_Index'] = (
    tableau_df['Doctor_Availability'].rank(pct=True) * 0.5
    + tableau_df['Cleanliness_Score'].rank(pct=True) * 0.3
    + tableau_df['Doctor_Experience_Years'].rank(pct=True) * 0.2
).round(4)

display(tableau_df.head())

In [ ]:
def summary_table(data, group_cols):
    out = data.groupby(group_cols, dropna=False).agg(
        Patient_Volume=('Patient_ID', 'count'),
        Avg_Treatment_Cost=('Treatment_Cost', 'mean'),
        Median_Treatment_Cost=('Treatment_Cost', 'median'),
        Recovery_Rate=('Recovered_Flag', 'mean'),
        Mortality_Rate=('Deceased_Flag', 'mean'),
        Under_Treatment_Rate=('Under_Treatment_Flag', 'mean'),
        Insurance_Coverage_Rate=('Insured_Flag', 'mean'),
        Avg_Doctor_Availability=('Doctor_Availability', 'mean'),
        Avg_Cleanliness_Score=('Cleanliness_Score', 'mean'),
        Avg_Operational_Access_Index=('Operational_Access_Index', 'mean'),
    ).reset_index()

    rate_cols = ['Recovery_Rate', 'Mortality_Rate', 'Under_Treatment_Rate', 'Insurance_Coverage_Rate']
    out[rate_cols] = (out[rate_cols] * 100).round(2)
    numeric_cols = out.select_dtypes(include='number').columns
    out[numeric_cols] = out[numeric_cols].round(2)
    return out

summary_specs = {
    'tableau_hospital_summary.csv': ['Hospital', 'Hospital_Type'],
    'tableau_diagnosis_summary.csv': ['Diagnosis'],
    'tableau_region_summary.csv': ['Region'],
    'tableau_age_group_summary.csv': ['Age_Group'],
    'tableau_economic_status_summary.csv': ['Economic_Status'],
    'tableau_insurance_summary.csv': ['Insurance'],
}

for filename, group_cols in summary_specs.items():
    table = summary_table(tableau_df, group_cols)
    table.to_csv(ANALYSIS_DIR / filename, index=False)
    print('Saved', ANALYSIS_DIR / filename)
    display(table)

In [ ]:
output_path = PROCESSED_DIR / 'hospital_tableau_ready.csv'
tableau_df.to_csv(output_path, index=False)
print(f'Saved Tableau-ready fact table: {output_path}')
print(f'Rows: {len(tableau_df):,}')
print(f'Columns: {tableau_df.shape[1]:,}')

## Tableau Handoff

Recommended dashboard views:

1. Executive overview: patient volume, average cost, recovery rate, mortality rate, under-treatment rate.
2. Cost drivers: cost by diagnosis, hospital, age group, insurance, and economic status.
3. Outcome risk: outcome mix by diagnosis, age group, hospital, and economic status.
4. Operations: doctor availability, cleanliness, experience, and operational access index.

Use filters for hospital, diagnosis, region, age group, economic status, and insurance.